# Subject 01 Visual Brain Low-Level VGG Transformer


# 1. Train + Eval VGG Encoder
Load fMRI responses, read cached Brain-IT-style VGG-16-BN token targets, train a Brain-IT-style tokenizer/cross-attention transformer, and evaluate predicted VGG features.


## Setup
Mount Drive, import libraries, set paths, and define low-level VGG/DIP hyperparameters.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, re, glob, json, random, shutil, time, gc
from collections import defaultdict, OrderedDict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

BASE = "/content"
DRIVE_PROJECT_DIR = f"{BASE}/drive/Shareddrives/FMRI_Paper"
INPUT_DIR = f"{DRIVE_PROJECT_DIR}/inputs"
BETA_DIR = f"{BASE}/subject01_visual_brain_responses"
OUTPUT_DIR = f"{DRIVE_PROJECT_DIR}/outputs/lowlevel_vgg_bit_{time.strftime('%Y%m%d_%H%M%S')}"
SUBJECT = "subj01"

SEED = 0
VAL_FRAC = 0.05
BATCH_SIZE = 128
BIT_EPOCHS = 5
BIT_LR = 1e-4
BIT_DIM = 256
BIT_NUM_CLUSTERS = 64
BIT_NUM_HEADS = 4
BIT_SELF_ATTN_LAYERS = 3
BIT_CROSS_ATTN_LAYERS = 1
BIT_DROPOUT = 0.10
GMM_PCA_DIMS = 48
GMM_MAX_TRIALS = 4096
GMM_MAX_ITER = 200
VGG_IMAGE_SIZE = 112
VGG_LAYER_NAMES = ["1_2", "2_2", "3_3", "4_3", "5_3"]
VGG_TEMPERATURE = 0.07
VGG_INFO_NCE_WEIGHT = 1.0
VGG_MSE_WEIGHT = 1.0
VGG_TRAIN_TOKENS_PER_LAYER = {"1_2": 256, "2_2": 256, "3_3": 64, "4_3": 32, "5_3": 8}
DIP_IMAGE_SIZE = 112
DIP_ITERS = 2000
DIP_LR = 1e-2
DIP_INPUT_DIM = 32
DIP_INTERNAL_DIM = 128
DIP_EMA = 0.99
DIP_MAX_IMAGES = 1000
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
AMP_DTYPE = torch.bfloat16

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
if DEVICE.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True
os.makedirs(OUTPUT_DIR, exist_ok=True)

for name in ["subject01_visual_brain_responses"]:
    dst = f"{BASE}/{name}"
    if not os.path.exists(dst):
        shutil.copytree(f"{INPUT_DIR}/{name}", dst)

print(f"device={DEVICE} batch_size={BATCH_SIZE} amp={USE_AMP} amp_dtype={AMP_DTYPE} output={OUTPUT_DIR}")

## Load Data
Load fMRI tensors, hold out shared-1000 as final test, and keep image ids so VGG targets can be extracted from the NSD stimulus image.

In [ ]:
import urllib.request
from scipy.io import loadmat

def session_id(path):
    return int(re.findall(r"\d+", os.path.basename(path))[-1])

beta_sess = {session_id(p): p for p in glob.glob(f"{BETA_DIR}/{SUBJECT}_visualroi_session*.pt")}
sessions = sorted(beta_sess)
assert sessions, "No fMRI sessions found."

betas = torch.cat([torch.load(beta_sess[s], map_location="cpu").float() for s in tqdm(sessions, desc="load fMRI")])

EXP = f"{BASE}/nsd_expdesign.mat"
if not os.path.exists(EXP):
    urllib.request.urlretrieve("https://natural-scenes-dataset.s3.amazonaws.com/nsddata/experiments/nsd/nsd_expdesign.mat", EXP)

mat = loadmat(EXP)
masterordering = mat["masterordering"].reshape(-1).astype(np.int64) - 1
subjectim = mat["subjectim"].astype(np.int64) - 1
imgbrick_ids = subjectim[int(SUBJECT[-2:]) - 1, masterordering]
shared_ids = set(mat["sharedix"].reshape(-1).astype(np.int64) - 1)

def image_id_for_concat_trial(gidx):
    session = sessions[int(gidx) // 750]
    offset = int(gidx) % 750
    return int(imgbrick_ids[(session - 1) * 750 + offset])

img_of = np.array([image_id_for_concat_trial(gidx) for gidx in range(len(betas))], dtype=np.int64)
is_shared = np.array([int(img_id) in shared_ids for img_id in img_of])

nonshared_img_ids = np.array(sorted(set(img_of[~is_shared])), dtype=np.int64)
image_perm = np.random.RandomState(SEED).permutation(nonshared_img_ids)
n_val_images = int(VAL_FRAC * len(image_perm))
val_img_ids = set(map(int, image_perm[:n_val_images]))
train_img_ids = set(map(int, image_perm[n_val_images:]))
in_val = np.array([int(img_id) in val_img_ids for img_id in img_of])
train_idx = torch.from_numpy(np.where(~is_shared & ~in_val)[0]).long()
val_idx = torch.from_numpy(np.where(~is_shared & in_val)[0]).long()

shared_groups = defaultdict(list)
for gidx, img_id in enumerate(img_of):
    if int(img_id) in shared_ids:
        shared_groups[int(img_id)].append(gidx)
shared_img_ids = list(shared_groups)
shared_eval_betas = torch.stack([betas[idxs].mean(0) for idxs in shared_groups.values()])

beta_mean = betas[train_idx].mean(0)
beta_std = betas[train_idx].std(0) + 1e-6
x = (betas - beta_mean) / beta_std
shared_eval_x = (shared_eval_betas - beta_mean) / beta_std
INPUT_DIM = x.shape[1]

train_loader = DataLoader(TensorDataset(x[train_idx], torch.from_numpy(img_of[train_idx.numpy()]).long()), batch_size=BATCH_SIZE, shuffle=True, generator=torch.Generator().manual_seed(SEED), pin_memory=True)
train_eval_loader = DataLoader(TensorDataset(x[train_idx], torch.from_numpy(img_of[train_idx.numpy()]).long()), batch_size=BATCH_SIZE, pin_memory=True)
val_loader = DataLoader(TensorDataset(x[val_idx], torch.from_numpy(img_of[val_idx.numpy()]).long()), batch_size=BATCH_SIZE, pin_memory=True)
test_loader = DataLoader(TensorDataset(shared_eval_x, torch.tensor(shared_img_ids, dtype=torch.long)), batch_size=BATCH_SIZE, pin_memory=True)

print(f"sessions={sessions}")
print(f"betas={tuple(betas.shape)} input_dim={INPUT_DIM}")
print(f"train_trials={len(train_idx)} val_trials={len(val_idx)} shared_test_images={len(shared_img_ids)}")

## Fit Voxel-to-Cluster GMM
Cluster subject voxels into Brain-IT-style brain-token neighborhoods. The saved mapping is reused on later runs.


In [ ]:
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture

V2C_PATH = f"{INPUT_DIR}/subject01_v2c_{BIT_NUM_CLUSTERS}_gmm.npy"
V2C_META_PATH = f"{INPUT_DIR}/subject01_v2c_{BIT_NUM_CLUSTERS}_gmm_meta.json"

def fit_or_load_voxel_gmm():
    if os.path.exists(V2C_PATH):
        labels = np.load(V2C_PATH).astype(np.int64)
        if labels.shape != (INPUT_DIM,):
            raise ValueError(f"V2C mapping has shape {labels.shape}, expected {(INPUT_DIM,)}: {V2C_PATH}")
        print(f"loaded voxel-to-cluster mapping: {V2C_PATH}")
        return labels

    rng = np.random.RandomState(SEED)
    trial_rows = train_idx.numpy()
    if len(trial_rows) > GMM_MAX_TRIALS:
        trial_rows = rng.choice(trial_rows, size=GMM_MAX_TRIALS, replace=False)
    voxel_profiles = x[torch.from_numpy(trial_rows)].T.numpy().astype(np.float32)
    pca_dims = min(GMM_PCA_DIMS, voxel_profiles.shape[0] - 1, voxel_profiles.shape[1] - 1)
    if pca_dims < 2:
        raise ValueError(f"Not enough data for voxel GMM PCA: profiles={voxel_profiles.shape}")

    pca = PCA(n_components=pca_dims, random_state=SEED, svd_solver="randomized")
    voxel_latents = pca.fit_transform(voxel_profiles)
    gmm = GaussianMixture(
        n_components=BIT_NUM_CLUSTERS,
        covariance_type="diag",
        reg_covar=1e-4,
        max_iter=GMM_MAX_ITER,
        random_state=SEED,
        verbose=1,
    )
    labels = gmm.fit_predict(voxel_latents).astype(np.int64)
    np.save(V2C_PATH, labels)
    meta = {
        "input_dim": int(INPUT_DIM),
        "num_clusters": int(BIT_NUM_CLUSTERS),
        "pca_dims": int(pca_dims),
        "gmm_trials": int(len(trial_rows)),
        "cluster_counts": np.bincount(labels, minlength=BIT_NUM_CLUSTERS).astype(int).tolist(),
    }
    with open(V2C_META_PATH, "w") as f:
        json.dump(meta, f, indent=2)
    print(f"saved voxel-to-cluster mapping: {V2C_PATH}")
    return labels

voxel_to_cluster = fit_or_load_voxel_gmm()
cluster_counts = np.bincount(voxel_to_cluster, minlength=BIT_NUM_CLUSTERS)
empty_clusters = np.where(cluster_counts == 0)[0].tolist()
print(f"clusters={BIT_NUM_CLUSTERS} min={cluster_counts.min()} median={int(np.median(cluster_counts))} max={cluster_counts.max()} empty={empty_clusters[:10]}")


## Brain-IT VGG Target Extractor
Use pretrained VGG-16-BN on 112x112 images and tokenize layers `1_2`, `2_2`, `3_3`, `4_3`, and `5_3` using the Brain-IT low-level layout.

In [ ]:
from torchvision import models

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
VGG_LAYER_INDICES = {"1_2": 5, "2_2": 12, "3_3": 22, "4_3": 32, "5_3": 42}
VGG_TOKEN_SPECS = {
    "1_2": (56 * 56, 64 * 4),
    "2_2": (55 * 55, 128 * 4),
    "3_3": (28 * 28, 256),
    "4_3": (14 * 14, 512),
    "5_3": (7 * 7, 512),
}

class BrainITVGGFeatures(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = models.vgg16_bn(weights=models.VGG16_BN_Weights.IMAGENET1K_V1).features.eval()
        for p in self.features.parameters():
            p.requires_grad_(False)

    def forward(self, x):
        out = {}
        h = x
        for idx, layer in enumerate(self.features):
            h = layer(h)
            for name, layer_idx in VGG_LAYER_INDICES.items():
                if idx == layer_idx:
                    out[name] = self.tokenize(name, h)
        return out

    @staticmethod
    def tokenize(name, feat):
        if name == "1_2":
            patches = F.unfold(feat, kernel_size=2, stride=2)
            return patches.transpose(1, 2).contiguous()
        if name == "2_2":
            patches = F.unfold(feat, kernel_size=2, stride=1)
            return patches.transpose(1, 2).contiguous()
        return feat.flatten(2).transpose(1, 2).contiguous()

vgg_target = BrainITVGGFeatures().to(DEVICE).eval()
for name, spec in VGG_TOKEN_SPECS.items():
    print(name, "spec", spec)

## Load Precomputed VGG Targets
Read existing `subject01_vgg_lowlevel_features` shards and rebuild shard-grouped loaders. No image-pixel HDF5 is needed.

In [ ]:
VGG_TARGET_DIR = f"{INPUT_DIR}/subject01_vgg_lowlevel_features"
VGG_LOCAL_TARGET_DIR = f"{BASE}/subject01_vgg_lowlevel_features"
VGG_READ_TARGET_DIR = VGG_LOCAL_TARGET_DIR if os.path.exists(f"{VGG_LOCAL_TARGET_DIR}/metadata.json") else VGG_TARGET_DIR
VGG_CACHE_DTYPE = torch.float16
VGG_CACHE_MAX_SHARDS_IN_RAM = 8
VGG_CACHE_USE_MMAP = True

metadata_path = f"{VGG_READ_TARGET_DIR}/metadata.json"
if not os.path.exists(metadata_path):
    raise FileNotFoundError(f"Missing precomputed VGG target metadata: {metadata_path}")

print(f"reading VGG targets from {VGG_READ_TARGET_DIR}")
with open(metadata_path) as f:
    _vgg_manifest = json.load(f)

_vgg_cache_lookup = {}
for shard in _vgg_manifest["shards"]:
    shard_path = f"{VGG_READ_TARGET_DIR}/{shard['file']}"
    if not os.path.exists(shard_path):
        raise FileNotFoundError(f"Missing VGG target shard: {shard_path}")
    for row, img_id in enumerate(shard["img_ids"]):
        _vgg_cache_lookup[int(img_id)] = (shard_path, row)

_vgg_shard_lru = OrderedDict()
_vgg_cache_stats = {"loads": 0, "load_seconds": 0.0}

def _load_vgg_shard(path):
    if path in _vgg_shard_lru:
        _vgg_shard_lru.move_to_end(path)
        return _vgg_shard_lru[path]
    t0 = time.time()
    try:
        shard = torch.load(path, map_location="cpu", mmap=VGG_CACHE_USE_MMAP)
    except TypeError:
        shard = torch.load(path, map_location="cpu")
    _vgg_cache_stats["loads"] += 1
    _vgg_cache_stats["load_seconds"] += time.time() - t0
    _vgg_shard_lru[path] = shard
    evicted = False
    while len(_vgg_shard_lru) > VGG_CACHE_MAX_SHARDS_IN_RAM:
        _vgg_shard_lru.popitem(last=False)
        evicted = True
    if evicted:
        gc.collect()
    return shard

def vgg_targets_for_ids(img_ids, token_indices_by_layer=None):
    ids = [int(i) for i in img_ids.detach().cpu().tolist()]
    missing = [i for i in ids if i not in _vgg_cache_lookup]
    if missing:
        raise KeyError(f"Missing cached VGG targets for image ids: {missing[:10]}")

    cpu_token_indices = {}
    for name in VGG_LAYER_NAMES:
        if token_indices_by_layer is not None and name in token_indices_by_layer:
            cpu_token_indices[name] = token_indices_by_layer[name].detach().cpu().long()

    out = {}
    for name in VGG_LAYER_NAMES:
        num_tokens, token_dim = VGG_TOKEN_SPECS[name]
        if name in cpu_token_indices:
            num_tokens = int(cpu_token_indices[name].numel())
        out[name] = torch.empty((len(ids), num_tokens, token_dim), dtype=VGG_CACHE_DTYPE)

    by_shard = defaultdict(list)
    for out_row, img_id in enumerate(ids):
        shard_path, shard_row = _vgg_cache_lookup[img_id]
        by_shard[shard_path].append((out_row, shard_row))
    for shard_path, pairs in by_shard.items():
        shard = _load_vgg_shard(shard_path)
        out_rows = [p[0] for p in pairs]
        shard_rows = [p[1] for p in pairs]
        row_idx = torch.as_tensor(shard_rows, dtype=torch.long)
        for name in VGG_LAYER_NAMES:
            features = shard["features"][name]
            if name in cpu_token_indices:
                tok_idx = cpu_token_indices[name]
                values = features[row_idx[:, None], tok_idx[None, :], :]
            else:
                values = features[row_idx]
            out[name][out_rows] = values
    return {name: value.to(DEVICE, non_blocking=True).float() for name, value in out.items()}

class VGGShardBatchSampler(torch.utils.data.Sampler):
    def __init__(self, img_ids, batch_size, shuffle=False, seed=0):
        self.img_ids = [int(i) for i in img_ids]
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.seed = seed
        self.epoch = 0
        self.groups = defaultdict(list)
        for row, img_id in enumerate(self.img_ids):
            self.groups[_vgg_cache_lookup[img_id][0]].append(row)

    def __iter__(self):
        rng = np.random.RandomState(self.seed + self.epoch)
        shard_paths = list(self.groups)
        if self.shuffle:
            rng.shuffle(shard_paths)
        batch = []
        for shard_path in shard_paths:
            rows = np.array(self.groups[shard_path], dtype=np.int64)
            if self.shuffle:
                rng.shuffle(rows)
            for row in rows.tolist():
                batch.append(row)
                if len(batch) == self.batch_size:
                    yield batch
                    batch = []
        if batch:
            yield batch
        self.epoch += 1

    def __len__(self):
        return (len(self.img_ids) + self.batch_size - 1) // self.batch_size

def make_cached_vgg_loader(x_tensor, img_id_tensor, batch_size, shuffle=False):
    ds = TensorDataset(x_tensor, img_id_tensor)
    sampler = VGGShardBatchSampler(img_id_tensor.tolist(), batch_size, shuffle=shuffle, seed=SEED)
    return DataLoader(ds, batch_sampler=sampler, pin_memory=True)

train_img_id_tensor = torch.from_numpy(img_of[train_idx.numpy()]).long()
val_img_id_tensor = torch.from_numpy(img_of[val_idx.numpy()]).long()
test_img_id_tensor = torch.tensor(shared_img_ids, dtype=torch.long)
train_loader = make_cached_vgg_loader(x[train_idx], train_img_id_tensor, BATCH_SIZE, shuffle=True)
train_eval_loader = make_cached_vgg_loader(x[train_idx], train_img_id_tensor, BATCH_SIZE, shuffle=False)
val_loader = make_cached_vgg_loader(x[val_idx], val_img_id_tensor, BATCH_SIZE, shuffle=False)
test_loader = make_cached_vgg_loader(shared_eval_x, test_img_id_tensor, BATCH_SIZE, shuffle=False)

sample_targets = vgg_targets_for_ids(torch.tensor(shared_img_ids[:2]))
for name in VGG_LAYER_NAMES:
    print(name, tuple(sample_targets[name].shape))
print(f"rebuilt shard-grouped loaders: train_batches={len(train_loader)} val_batches={len(val_loader)} test_batches={len(test_loader)}")

## Brain-IT Tokenizer + Cross-Attention Transformer
Graph-attend voxels into cluster tokens, contextualize brain tokens, then let per-VGG-layer query tokens cross-attend to the brain representation.


In [ ]:
class BrainTokenizerGAT(nn.Module):
    def __init__(self, input_dim, cluster_ids, num_clusters, dim):
        super().__init__()
        self.input_dim = input_dim
        self.num_clusters = num_clusters
        self.dim = dim
        self.voxel_embed = nn.Parameter(torch.randn(input_dim, dim) * 0.02)
        self.cluster_embed = nn.Parameter(torch.randn(num_clusters, dim) * 0.02)
        self.to_q = nn.Linear(dim, dim, bias=False)
        self.to_k = nn.Linear(dim, dim, bias=False)
        self.to_v = nn.Linear(dim, dim, bias=False)
        self.out_norm = nn.LayerNorm(dim)

        cluster_ids = torch.as_tensor(cluster_ids, dtype=torch.long)
        for k in range(num_clusters):
            idx = torch.where(cluster_ids == k)[0]
            if idx.numel() == 0:
                idx = torch.tensor([0], dtype=torch.long)
            self.register_buffer(f"cluster_idx_{k}", idx)

    def forward(self, x):
        b = x.size(0)
        tokens = []
        scale = self.dim ** -0.5
        for k in range(self.num_clusters):
            idx = getattr(self, f"cluster_idx_{k}")
            voxel_h = x[:, idx].unsqueeze(-1) * self.voxel_embed[idx].unsqueeze(0)
            cluster_h = self.cluster_embed[k].view(1, 1, -1).expand(b, 1, -1)
            q = self.to_q(cluster_h)
            k_tokens = self.to_k(voxel_h)
            v_tokens = self.to_v(voxel_h)
            attn = torch.softmax(torch.matmul(q, k_tokens.transpose(1, 2)) * scale, dim=-1)
            token = torch.matmul(attn, v_tokens).squeeze(1) + cluster_h.squeeze(1)
            tokens.append(token)
        return self.out_norm(torch.stack(tokens, dim=1))

class CrossAttentionBlock(nn.Module):
    def __init__(self, dim, heads, dropout):
        super().__init__()
        self.q_norm = nn.LayerNorm(dim)
        self.mem_norm = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, dropout=dropout, batch_first=True)
        self.ffn_norm = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(
            nn.Linear(dim, dim * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim * 4, dim),
            nn.Dropout(dropout),
        )

    def forward(self, query, memory):
        q = self.q_norm(query)
        mem = self.mem_norm(memory)
        attended, _ = self.attn(q, mem, mem, need_weights=False)
        query = query + attended
        return query + self.ffn(self.ffn_norm(query))

class BrainITVGGTransformer(nn.Module):
    def __init__(self, input_dim, cluster_ids, token_specs):
        super().__init__()
        self.tokenizer = BrainTokenizerGAT(input_dim, cluster_ids, BIT_NUM_CLUSTERS, BIT_DIM)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=BIT_DIM,
            nhead=BIT_NUM_HEADS,
            dim_feedforward=BIT_DIM * 4,
            dropout=BIT_DROPOUT,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.brain_encoder = nn.TransformerEncoder(enc_layer, num_layers=BIT_SELF_ATTN_LAYERS)
        self.query_tokens = nn.ParameterDict({
            name: nn.Parameter(torch.randn(num_tokens, BIT_DIM) * 0.02)
            for name, (num_tokens, _) in token_specs.items()
        })
        self.cross_blocks = nn.ModuleList([
            CrossAttentionBlock(BIT_DIM, BIT_NUM_HEADS, BIT_DROPOUT)
            for _ in range(BIT_CROSS_ATTN_LAYERS)
        ])
        self.out = nn.ModuleDict({
            name: nn.Linear(BIT_DIM, token_dim)
            for name, (_, token_dim) in token_specs.items()
        })

    def forward(self, x, token_indices_by_layer=None):
        brain_tokens = self.brain_encoder(self.tokenizer(x))
        preds = {}
        b = x.size(0)
        for name in VGG_LAYER_NAMES:
            q_tokens = self.query_tokens[name]
            if token_indices_by_layer is not None and name in token_indices_by_layer:
                q_tokens = q_tokens[token_indices_by_layer[name]]
            query = q_tokens.unsqueeze(0).expand(b, -1, -1)
            for block in self.cross_blocks:
                query = block(query, brain_tokens)
            preds[name] = self.out[name](query)
        return preds

model = BrainITVGGTransformer(INPUT_DIM, voxel_to_cluster, VGG_TOKEN_SPECS).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=BIT_LR, weight_decay=1e-4)
n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"parameters={n_params:,}")


## Train VGG Encoder
Train with Brain-IT-style per-layer InfoNCE over VGG tokens, and report raw VGG MSE/cosine as diagnostics.

In [ ]:
def sample_vgg_token_indices(device, deterministic=False, seed=None):
    out = {}
    rng = np.random.RandomState(SEED if seed is None else seed)
    for name in VGG_LAYER_NAMES:
        n_tokens = VGG_TOKEN_SPECS[name][0]
        n_keep = min(VGG_TRAIN_TOKENS_PER_LAYER[name], n_tokens)
        if deterministic:
            idx = torch.linspace(0, n_tokens - 1, steps=n_keep).long()
        else:
            idx = torch.from_numpy(np.sort(rng.choice(n_tokens, size=n_keep, replace=False))).long()
        out[name] = idx.to(device)
    return out

def layer_infonce(pred, target, temperature=VGG_TEMPERATURE):
    b, t, _ = pred.shape
    pred = F.normalize(pred, dim=-1)
    target = F.normalize(target, dim=-1)
    logits = torch.einsum("btd,ctd->tbc", pred, target) / temperature
    labels = torch.arange(b, device=pred.device).expand(t, b).reshape(-1)
    return F.cross_entropy(logits.reshape(t * b, b), labels)

def vgg_loss(preds, targets, token_indices_by_layer=None):
    parts = {}
    loss = torch.zeros((), device=next(iter(preds.values())).device)
    for name in VGG_LAYER_NAMES:
        target = targets[name]
        if token_indices_by_layer is not None and name in token_indices_by_layer:
            target = target[:, token_indices_by_layer[name]]
        cont = layer_infonce(preds[name], target)
        mse = F.mse_loss(preds[name], target)
        cos = F.cosine_similarity(preds[name].flatten(1), target.flatten(1), dim=1).mean()
        layer_loss = VGG_INFO_NCE_WEIGHT * cont + VGG_MSE_WEIGHT * mse
        parts[f"{name}_infonce"] = cont
        parts[f"{name}_mse"] = mse
        parts[f"{name}_cos"] = cos
        parts[f"{name}_loss"] = layer_loss
        loss = loss + layer_loss
    parts["loss"] = loss / len(VGG_LAYER_NAMES)
    return parts

@torch.no_grad()
def evaluate_vgg_encoder(loader, max_batches=None):
    model.eval()
    totals = defaultdict(float)
    seen = 0
    eval_indices = sample_vgg_token_indices(DEVICE, deterministic=True)
    for batch_i, (xb, img_ids) in enumerate(tqdm(loader, desc="eval vgg", leave=False)):
        if max_batches is not None and batch_i >= max_batches:
            break
        xb = xb.to(DEVICE, non_blocking=True)
        targets = vgg_targets_for_ids(img_ids, token_indices_by_layer=eval_indices)
        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
            preds = model(xb, token_indices_by_layer=eval_indices)
            parts = vgg_loss(preds, targets)
        n = xb.size(0)
        for k, v in parts.items():
            totals[k] += n * float(v.detach().cpu())
        seen += n
    return {k: v / max(seen, 1) for k, v in totals.items()}

# The optimizer minimizes this VGG-only objective; lower loss means stronger low-level reconstruction targets.
vgg_history = []
best_val = float("inf")
for epoch in range(1, BIT_EPOCHS + 1):
    model.train()
    totals = defaultdict(float)
    seen = 0
    epoch_token_indices = sample_vgg_token_indices(DEVICE, deterministic=False, seed=SEED + epoch)
    pbar = tqdm(train_loader, desc=f"vgg bit {epoch:03d}/{BIT_EPOCHS}", leave=False)
    for xb, img_ids in pbar:
        batch_t0 = time.time()
        xb = xb.to(DEVICE, non_blocking=True)
        token_indices = epoch_token_indices
        target_t0 = time.time()
        targets = vgg_targets_for_ids(img_ids, token_indices_by_layer=token_indices)
        target_s = time.time() - target_t0
        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
            parts = vgg_loss(model(xb, token_indices_by_layer=token_indices), targets)
            loss = parts["loss"]
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        step_s = time.time() - batch_t0
        n = xb.size(0)
        for k, v in parts.items():
            totals[k] += n * float(v.detach().cpu())
        seen += n
        pbar.set_postfix(loss=loss.item(), target_s=f"{target_s:.2f}", step_s=f"{step_s:.2f}", cache=len(_vgg_shard_lru))

    row = {f"train_{k}": v / seen for k, v in totals.items()}
    tqdm.write(f"vgg_cache loads={_vgg_cache_stats['loads']} cached={len(_vgg_shard_lru)} load_seconds={_vgg_cache_stats['load_seconds']:.1f}")
    val_scores = evaluate_vgg_encoder(val_loader)
    row.update({f"val_{k}": v for k, v in val_scores.items()})
    row["epoch"] = epoch
    vgg_history.append(row)
    tqdm.write(f"epoch={epoch:03d} train_loss={row['train_loss']:.4f} val_loss={row['val_loss']:.4f}")

    if row["val_loss"] < best_val:
        best_val = row["val_loss"]
        torch.save({
            "model": model.state_dict(),
            "beta_mean": beta_mean,
            "beta_std": beta_std,
            "token_specs": VGG_TOKEN_SPECS,
            "layer_names": VGG_LAYER_NAMES,
            "voxel_to_cluster": voxel_to_cluster,
            "bit_dim": BIT_DIM,
            "bit_num_clusters": BIT_NUM_CLUSTERS,
            "bit_num_heads": BIT_NUM_HEADS,
            "bit_self_attn_layers": BIT_SELF_ATTN_LAYERS,
            "bit_cross_attn_layers": BIT_CROSS_ATTN_LAYERS,
            "loss": "VGG InfoNCE + VGG MSE",
        }, f"{OUTPUT_DIR}/best_vgg_bit_encoder.pt")

with open(f"{OUTPUT_DIR}/vgg_bit_history.json", "w") as f:
    json.dump(vgg_history, f, indent=2)
print(f"saved={OUTPUT_DIR}/best_vgg_bit_encoder.pt")


## Evaluate VGG Encoder
Reload the best encoder and score train/validation/shared-1000 VGG prediction quality.

In [ ]:
ckpt = torch.load(f"{OUTPUT_DIR}/best_vgg_bit_encoder.pt", map_location=DEVICE)
model.load_state_dict(ckpt["model"])
final_train = evaluate_vgg_encoder(train_eval_loader, max_batches=50)
final_val = evaluate_vgg_encoder(val_loader)
final_test = evaluate_vgg_encoder(test_loader)

summary = {"train_sampled": final_train, "val": final_val, "shared1000": final_test}
with open(f"{OUTPUT_DIR}/vgg_encoder_eval.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))

plt.plot([h["epoch"] for h in vgg_history], [h["train_loss"] for h in vgg_history], label="train")
plt.plot([h["epoch"] for h in vgg_history], [h["val_loss"] for h in vgg_history], label="val")
plt.xlabel("epoch")
plt.ylabel("VGG Loss")
plt.legend()
plt.savefig(f"{OUTPUT_DIR}/vgg_bit_train_val_loss.png", dpi=180, bbox_inches="tight")
plt.show()

# 2. Invert VGG To Get Image
Freeze the VGG encoder predictions and optimize a Deep Image Prior U-Net per shared-1000 sample so generated VGG features match the predicted features.

## Deep Image Prior Inversion
Use the Brain-IT low-level recipe: fixed noise input, convolutional U-Net image prior, VGG feature matching loss, 2K optimization steps, and EMA-smoothed output.

In [ ]:
from torchvision.utils import save_image

class DIPUNet(nn.Module):
    def __init__(self, in_ch=DIP_INPUT_DIM, base=DIP_INTERNAL_DIM):
        super().__init__()
        self.enc1 = nn.Sequential(nn.Conv2d(in_ch, base, 3, padding=1), nn.LeakyReLU(0.2, inplace=True), nn.Conv2d(base, base, 3, padding=1), nn.LeakyReLU(0.2, inplace=True))
        self.enc2 = nn.Sequential(nn.Conv2d(base, base * 2, 3, padding=1), nn.LeakyReLU(0.2, inplace=True), nn.Conv2d(base * 2, base * 2, 3, padding=1), nn.LeakyReLU(0.2, inplace=True))
        self.enc3 = nn.Sequential(nn.Conv2d(base * 2, base * 4, 3, padding=1), nn.LeakyReLU(0.2, inplace=True), nn.Conv2d(base * 4, base * 4, 3, padding=1), nn.LeakyReLU(0.2, inplace=True))
        self.pool = nn.AvgPool2d(2)
        self.up2 = nn.Conv2d(base * 4, base * 2, 1)
        self.dec2 = nn.Sequential(nn.Conv2d(base * 4, base * 2, 3, padding=1), nn.LeakyReLU(0.2, inplace=True), nn.Conv2d(base * 2, base * 2, 3, padding=1), nn.LeakyReLU(0.2, inplace=True))
        self.up1 = nn.Conv2d(base * 2, base, 1)
        self.dec1 = nn.Sequential(nn.Conv2d(base * 2, base, 3, padding=1), nn.LeakyReLU(0.2, inplace=True), nn.Conv2d(base, base, 3, padding=1), nn.LeakyReLU(0.2, inplace=True))
        self.out = nn.Conv2d(base, 3, 1)

    def forward(self, z):
        e1 = self.enc1(z)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        u2 = F.interpolate(self.up2(e3), size=e2.shape[-2:], mode="bilinear", align_corners=False)
        d2 = self.dec2(torch.cat([u2, e2], dim=1))
        u1 = F.interpolate(self.up1(d2), size=e1.shape[-2:], mode="bilinear", align_corners=False)
        d1 = self.dec1(torch.cat([u1, e1], dim=1))
        return torch.sigmoid(self.out(d1))

def imagenet_normalize_01(img01):
    mean = IMAGENET_MEAN.to(img01.device)
    std = IMAGENET_STD.to(img01.device)
    return (img01 - mean) / std

def dip_vgg_loss(img01, target_tokens):
    pred_tokens = vgg_target(imagenet_normalize_01(img01))
    return sum(F.mse_loss(pred_tokens[name], target_tokens[name]) for name in VGG_LAYER_NAMES) / len(VGG_LAYER_NAMES)

@torch.no_grad()
def predict_vgg_tokens(xb):
    model.eval()
    return {k: v.detach() for k, v in model(xb.to(DEVICE, non_blocking=True)).items()}

def invert_one_vgg(target_tokens, seed=0, iters=DIP_ITERS):
    torch.manual_seed(seed)
    dip = DIPUNet().to(DEVICE).train()
    z = torch.randn(1, DIP_INPUT_DIM, DIP_IMAGE_SIZE, DIP_IMAGE_SIZE, device=DEVICE)
    opt = torch.optim.Adam(dip.parameters(), lr=DIP_LR)
    ema = None
    pbar = tqdm(range(iters), desc="DIP", leave=False)
    for _ in pbar:
        img = dip(z)
        loss = dip_vgg_loss(img, target_tokens)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        with torch.no_grad():
            ema = img.detach() if ema is None else DIP_EMA * ema + (1 - DIP_EMA) * img.detach()
        pbar.set_postfix(loss=float(loss.detach().cpu()))
    return ema.clamp(0, 1).cpu()

model.eval()
for p in model.parameters():
    p.requires_grad_(False)

DIP_RECONS_PATH = f"{OUTPUT_DIR}/lowlevel_vgg_dip_recons.pt"
DIP_IDS_PATH = f"{OUTPUT_DIR}/lowlevel_vgg_dip_img_ids.json"

def run_dip_shared1000(max_images=DIP_MAX_IMAGES):
    if os.path.exists(DIP_RECONS_PATH):
        print(f"loading cached DIP reconstructions: {DIP_RECONS_PATH}")
        return torch.load(DIP_RECONS_PATH, map_location="cpu")
    recons = []
    n_done = 0
    for xb, img_ids in tqdm(test_loader, desc="shared-1000 DIP batches"):
        b = xb.size(0)
        for j in range(b):
            if n_done >= max_images:
                break
            target = predict_vgg_tokens(xb[j:j + 1])
            recon = invert_one_vgg(target, seed=SEED + n_done)
            recons.append(recon.squeeze(0))
            n_done += 1
        if n_done >= max_images:
            break
    recons = torch.stack(recons).clamp(0, 1)
    torch.save(recons, DIP_RECONS_PATH)
    with open(DIP_IDS_PATH, "w") as f:
        json.dump([int(i) for i in shared_img_ids[:len(recons)]], f)
    print(f"saved={DIP_RECONS_PATH} shape={tuple(recons.shape)}")
    return recons

lowlevel_recons = run_dip_shared1000()
print("lowlevel_recons", tuple(lowlevel_recons.shape))

# 3. 1000 Eval Of Inverted VGG Low-Level Representation
Evaluate the DIP-inverted low-level reconstructions against the held-out shared-1000 images.

## Shared-1000 VGG Reconstruction Metrics
Evaluate DIP outputs by comparing their VGG features to the precomputed ground-truth VGG targets.

In [ ]:
lowlevel_recons = torch.load(DIP_RECONS_PATH, map_location="cpu")
EVAL_N = min(1000, len(lowlevel_recons), len(shared_img_ids))
recons_eval = lowlevel_recons[:EVAL_N].float().clamp(0, 1)
ids_eval = torch.tensor(shared_img_ids[:EVAL_N], dtype=torch.long)

@torch.no_grad()
def eval_vgg_reconstruction_metrics(recons, img_ids, batch_size=16):
    mse_totals = defaultdict(float)
    cos_totals = defaultdict(float)
    seen = 0
    for start in tqdm(range(0, len(recons), batch_size), desc="eval VGG recon"):
        rec = imagenet_normalize_01(recons[start:start + batch_size].to(DEVICE))
        pred_tokens = vgg_target(rec)
        true_tokens = vgg_targets_for_ids(img_ids[start:start + batch_size])
        n = len(rec)
        for name in VGG_LAYER_NAMES:
            mse_totals[name] += n * float(F.mse_loss(pred_tokens[name], true_tokens[name]).cpu())
            cos_totals[name] += n * float(F.cosine_similarity(pred_tokens[name].flatten(1), true_tokens[name].flatten(1), dim=1).mean().cpu())
        seen += n
    out = {"N": seen}
    for name in VGG_LAYER_NAMES:
        out[f"VGG_{name}_MSE"] = mse_totals[name] / seen
        out[f"VGG_{name}_Cosine"] = cos_totals[name] / seen
    return out

metrics = eval_vgg_reconstruction_metrics(recons_eval, ids_eval)
metrics_df = pd.DataFrame([metrics]).set_index("N")
display(metrics_df)
metrics_df.to_csv(f"{OUTPUT_DIR}/lowlevel_vgg_dip_shared1000_vgg_eval.csv")
print(f"saved={OUTPUT_DIR}/lowlevel_vgg_dip_shared1000_vgg_eval.csv")

## Preview Grid
Save an actual-vs-low-level reconstruction grid from the DIP outputs.

In [ ]:
PREVIEW_GRID_N = 48
PREVIEW_GRID_COLS = 8
PREVIEW_GRID_PATH = f"{OUTPUT_DIR}/lowlevel_vgg_dip_preview_grid.png"

n_show = min(PREVIEW_GRID_N, len(lowlevel_recons))
rng = np.random.RandomState(SEED)
show_idx = rng.choice(len(lowlevel_recons), size=n_show, replace=False)
cols = min(PREVIEW_GRID_COLS, n_show)
rows = int(np.ceil(n_show / cols))

fig, axes = plt.subplots(rows, cols, figsize=(2.4 * cols, 2.8 * rows), constrained_layout=True)
axes = np.atleast_1d(axes).reshape(rows, cols)
for ax in axes.ravel():
    ax.axis("off")
for k, idx in enumerate(show_idx):
    ax = axes[k // cols, k % cols]
    ax.imshow(lowlevel_recons[int(idx)].permute(1, 2, 0).clamp(0, 1))
    ax.set_title(f"shared id {shared_img_ids[int(idx)]}", fontsize=8)
    ax.axis("off")
fig.suptitle("VGG-DIP Low-Level Reconstructions", fontsize=14)
plt.savefig(PREVIEW_GRID_PATH, dpi=180, bbox_inches="tight")
plt.show()
print(f"saved={PREVIEW_GRID_PATH}")